# Exercise 8

In this exercise we will explore the use of the `COPY INTO`  commanc to create Delta Tables. Iceberg table creation will be implemented but not tested due to the Databricks Free limitations

In [0]:
%sql
-- Step 0: Volume creation
CREATE VOLUME IF NOT EXISTS workspace.default.copy_demo;

In [0]:
# Step 0.1: Input set up
dbutils.fs.mkdirs("/Volumes/workspace/default/copy_demo/input")

# Step 0.2: Sample JSON Files Creation
dbutils.fs.put(
    "/Volumes/workspace/default/copy_demo/input/batch1.json",
    """{"id": 1, "name": "Polnareff"}
{"id": 2, "name": "Kakyoin"}"""
)

Wrote 59 bytes.


True

In [0]:
%sql
-- Step 0.3: Delta Table Creation
CREATE TABLE IF NOT EXISTS workspace.default.copy_into_demo (
  id BIGINT,
  name STRING
);

In [0]:
%sql
-- Step 1: COPY INTO (Basic)
COPY INTO workspace.default.copy_into_demo
FROM '/Volumes/workspace/default/copy_demo/input'
FILEFORMAT = JSON;

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
2,2,0


In [0]:
%sql
-- Step 1.1: Validation
SELECT * FROM workspace.default.copy_into_demo;

id,name
1,Polnareff
2,Kakyoin


In [0]:
# Step 2: COPY INTO (Incremental)
dbutils.fs.put(
    "/Volumes/workspace/default/copy_demo/input/batch2.json",
    """{"id": 3, "name": "DIO"}"""
)

Wrote 24 bytes.


True

In [0]:
%sql
-- Step 2.1: Execute COPY INTO
COPY INTO workspace.default.copy_into_demo
FROM '/Volumes/workspace/default/copy_demo/input'
FILEFORMAT = JSON;

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
1,1,0


In [0]:
%sql
-- Step 2.2: Validate
SELECT * FROM workspace.default.copy_into_demo;

id,name
1,Polnareff
2,Kakyoin
3,DIO


In [0]:
%sql
-- Step 2.3: View Ingestion History
DESCRIBE HISTORY workspace.default.copy_into_demo;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-04-20T23:24:25.000Z,72040222688145,caer23.voltr@outlook.com,COPY INTO,Map(statsOnLoad -> true),null,List(1867190688964937),b3262355-a920-4f7d-a5c5-d83e179b10c4,0420-231655-2f1zeob1-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 827, numSkippedCorruptFiles -> 0)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-04-20T21:02:52.000Z,72040222688145,caer23.voltr@outlook.com,COPY INTO,Map(statsOnLoad -> true),null,List(1867190688964937),e739775a-2516-43c5-94a1-c46ee17e4a42,0420-202922-xhs2iq66-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 875, numSkippedCorruptFiles -> 0)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-04-20T20:49:02.000Z,72040222688145,caer23.voltr@outlook.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-a77d9740-47e4-499c-aaa5-2682290bc211"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-71f073b7-1792-4777-8adf-d61b3b2ccd60""}, statsOnLoad -> false)",null,List(1867190688964937),950feba7-2aca-4556-b02c-c42e8bbab039,0420-202922-xhs2iq66-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


# Iceberg Usage

This code will be the base to create Iceberg tables in Databricks, by definition it cannot be executed in Free edition, but it will when migrate to Pro. However is necessary to have it present for architecture development

In [0]:
%sql
-- Step 0: Catalog Cration via MANAGED LOCATION
CREATE CATALOG IF NOT EXISTS analytics
MANAGED LOCATION 's3://my-bucket/analytics/';

-- Step 0.1: Schema Creation
CREATE SCHEMA IF NOT EXISTS analytics.raw;

-- Step 1: Iceberg Table Creation
CREATE TABLE analytics.raw.events (
  event_id STRING,
  ts TIMESTAMP,
  payload STRING
)
USING ICEBERG;
    
-- Step 2: COPY INTO
-- USING COPY INTO from Volumes
COPY INTO analytics.raw.events
FROM '/Volumes/<PATH>/input'
FILEFORMAT = JSON;

-- USING COPY INTO from S3
COPY INTO analytics.raw.events
FROM 's3://my-bucket/events/'
FILEFORMAT = JSON;

-- USING INSERT
INSERT INTO analytics.raw.events VALUES
("e1", now(), "hello"),
("e2", now(), "world");

-- Step 3: Validate
SELECT * FROM analytics.raw.events;

-- EXTRA: Time Travel
SELECT * FROM analytics.raw.events VERSION AS OF 0;

In [0]:
## Using Spark DataFrame to insert data directly to the Iceberg Table
# ============================================================
# 1. Create a Spark DataFrame
# ============================================================

from pyspark.sql import Row
from pyspark.sql import functions as F

data = [
    Row(event_id="e100", ts="2024-01-01T10:00:00Z", value=10),
    Row(event_id="e101", ts="2024-01-01T11:00:00Z", value=20),
    Row(event_id="e102", ts="2024-01-01T12:00:00Z", value=30)
]

df = spark.createDataFrame(data).withColumn("ts", F.to_timestamp("ts"))

df.show()

(
    df.write
      .format("iceberg")     # Required for Iceberg tables
      .mode("append")        # append / overwrite / overwriteDynamic
      .saveAsTable("analytics.raw.iceberg_events")
)